In [2]:
# ! pip install langchain-chroma

### RAG

In [4]:
from getpass import getpass
import os

os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI API: ")

In [5]:
import json

with open("./router_agent_documents.json", "r") as f:
    knowledge_base = json.load(f)
knowledge_base[:3]

[{'text': 'Question: How do I integrate your AI product with my existing CRM system? Answer: You can integrate our AI product with your CRM using our API. Refer to the API documentation available on our website for step-by-step guidance.',
  'metadata': {'category': 'technical'}},
 {'text': 'Question: What programming languages are supported by your SDK? Answer: Our SDK supports Python, Java, and JavaScript. Additional language support is planned for future updates.',
  'metadata': {'category': 'technical'}},
 {'text': 'Question: Can your AI models run on-premise? Answer: Yes, our AI models can be deployed on-premise. We provide deployment guides for various environments.',
  'metadata': {'category': 'technical'}}]

In [8]:
from langchain_core.documents import Document
from tqdm import tqdm

processed_docs = []

for doc in tqdm(knowledge_base):
    metadata = doc["metadata"]
    data = doc["text"]
    processed_docs.append(Document(page_content=data,
                                   metadata=metadata))
    
    processed_docs[:3]

100%|██████████| 30/30 [00:00<00:00, 43419.30it/s]


In [9]:
from langchain_openai import OpenAIEmbeddings

openai_embed_model = OpenAIEmbeddings(model = "text-embedding-3-small")

In [10]:
from langchain_chroma import Chroma

kbase_db = Chroma.from_documents(documents=processed_docs,
                                 collection_name="knowledge_base",
                                 embedding=openai_embed_model,
                                 collection_metadata={"hnsw:space":"cosine"},
                                 persist_directory="./knowledge_base")

In [11]:
kbase_search = kbase_db.as_retriever(search_type = "similarity_score_threshold",
                                     search_kwargs = {"k":3, "score_threshold":0.2})

In [12]:
query = "What is the refund policy?"
metadata_filter = {"category" : "general"}
kbase_search.search_kwargs["filter"] = metadata_filter
kbase_search.invoke(query)

[Document(id='295a3ab6-653c-4268-9026-689b29375e52', metadata={'category': 'general'}, page_content='Question: What is your refund policy? Answer: We offer a 30-day money-back guarantee for all our products. Please contact support to initiate a refund.'),
 Document(id='7c5226b2-f391-490e-b5f5-6069c9e80b43', metadata={'category': 'general'}, page_content='Question: What is your policy for handling damaged hardware deliveries? Answer: If you receive damaged hardware, please report it within 48 hours to our support team. We will arrange for a replacement.'),
 Document(id='e90782cd-97d4-43a1-a1b6-843bd9f37d98', metadata={'category': 'general'}, page_content='Question: What is your shipping policy for hardware products? Answer: We provide free shipping for orders above $500. For orders below $500, a flat shipping fee of $20 applies. Shipping typically takes 5-7 business days.')]

### LangGraph State

In [14]:
from typing import TypedDict, Literal
from pydantic import BaseModel

class CustomerSupportState(TypedDict):
    customer_query: str 
    query_category: str 
    query_sentiment: str 
    final_response: str 

class QueryCategory(BaseModel):
    categorized_topic: Literal['Technical', 'Biling', 'General']

class QuerySentiment(BaseModel):
    sentiment: Literal['Positive', 'Neutral', 'negative']


In [15]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [17]:
def categorize_inquiry(support_state: CustomerSupportState) -> CustomerSupportState:
    """
    Classify the customer query into Technical, Billing, or General.
    """

    query = support_state["customer_query"]
    ROUTE_CATEGORY_PROMPT = """Act as a customer support agent trying to best categorize the customer query.
                               You are an agent for an AI products and hardware company.

                               Please read the customer query below and
                               determine the best category from the following list:

                               'Technical', 'Billing', or 'General'.

                               Remember:
                                - Technical queries will focus more on technical aspects like AI models, hardware, software related queries etc.
                                - General queries will focus more on general aspects like contacting support, finding things, policies etc.
                                - Billing queries will focus more on payment and purchase related aspects

                                Return just the category name (from one of the above)

                                Query:
                                {customer_query}
                            """
    
    prompt = ROUTE_CATEGORY_PROMPT.format(customer_query=query)
    route_category = llm.with_structured_output(QueryCategory).invoke(prompt)

    return {
        "query_category": route_category.categorized_topic
    }

In [18]:
categorize_inquiry({"customer_query": "Do you provide pretrained models?"})

{'query_category': 'Technical'}

In [19]:
categorize_inquiry({"customer_query": "what is your refund policy?"})

{'query_category': 'General'}

In [20]:
categorize_inquiry({"customer_query": "what payment methods are accepted?"})

{'query_category': 'Biling'}

In [24]:
def analyze_inquiry_sentiment(support_state:CustomerSupportState)->CustomerSupportState:

    query = support_state["customer_query"]
    SENTIMENT_CATEGORY_PROMPT = """Act as a customer support agent trying to best categorize the customer query's sentiment.
                                   You are an agent for an AI products and hardware company.

                                   Please read the customer query below,
                                   analyze its sentiment which should be one from the following list:

                                   'Positive', 'Neutral', or 'Negative'.

                                   Return just the sentiment (from one of the above)

                                   Query:
                                   {customer_query}
                                """
    
    prompt = SENTIMENT_CATEGORY_PROMPT.format(customer_query = query)
    sentiment_category = llm.with_structured_output(QuerySentiment).invoke(prompt)

    return {
        "query_sentiment": sentiment_category.sentiment
    }

In [25]:
analyze_inquiry_sentiment({"customer_query": "what is your refund policy?"})

{'query_sentiment': 'Neutral'}

In [26]:
analyze_inquiry_sentiment({"customer_query": "what is your refund policy? I am really fed up with this product and need to refund it"})

{'query_sentiment': 'negative'}